# Scenario - 4

In [1]:
import sys

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["DGLBACKEND"] = "pytorch"

import torch

DEVICE = torch.device("cuda:0")  # GPU 1 becomes cuda:0 now

print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))

NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition
(12, 0)


In [2]:

# import os
# from dotenv import load_dotenv
# from openai import OpenAI

# load_dotenv()

# api_key = os.getenv("DEEPSEEK_API")

# endpoint = os.getenv("END_POINT")
# MODEL_ID = "DeepSeek-V4-Flash"

# client = OpenAI(
#     base_url=f"{endpoint}/openai/v1/",
#     api_key=api_key,
# )

# response = client.chat.completions.create(
#     model=MODEL_ID,
#     messages=[
#         {
#             "role": "user",
#             "content": "What is the capital of France?"
#         }
#     ],
# )

# print(response.choices[0].message.content)

In [ ]:

import torch
from collections import defaultdict
import statistics as stats
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
 
from dotenv import load_dotenv
from huggingface_hub import login
 
load_dotenv()
token = os.getenv("HF_TOKEN")
login(token=token)


# MODEL_ID = "Qwen/Qwen3-14B"
# MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
# MODEL_ID = "mistralai--Mistral-7B-v0.1"
# MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"
# MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"   # gated -- must accept license + login() above
 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
 
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
 
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",       # spreads layers across available GPU(s), offloads to CPU if needed
    torch_dtype=torch.bfloat16,
)
model.eval()
print("Model loaded.")

/home/siu856622573/.conda/envs/py10_roy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   1%|          | 2/243 [00:00<00:22, 10.73it/s]/home/siu856622573/.conda/envs/py10_roy/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 243/243 [00:02<00:00, 87.32it/s]


Model loaded.


In [ ]:

# import os
# from dotenv import load_dotenv
# from openai import OpenAI

# load_dotenv()

# api_key = os.getenv("GPT_LUNA")

# endpoint = os.getenv("END_POINT")
# MODEL_ID = "GPT-5.6-LUNA"

# client = OpenAI(
#     base_url=f"{endpoint}/openai/v1/",
#     api_key=api_key,
# )

# response = client.chat.completions.create(
#     model=MODEL_ID,
#     messages=[
#         {
#             "role": "user",
#             "content": "What is the capital of France?"
#         }
#     ],
# )

# print(response.choices[0].message.content)

# Load Files

In [4]:
import os
import json
import re
import torch
import pandas as pd

  
ROI_NAMES_JSON = "../aal_roi_names.json"                # region-name legend, matches BLEG's "{template}" field
META_CSV = "../subject_summary.csv"


EDGE_LIST_NUMBERS = "../edge_list_top10pos_numbers.json"   # {subject_id: "Node feature:\n...\n\nEdge feature:\n..."}
ZSCORED_NUMBERS = "../zscore_top10pos_numbers.json"  

DATASET_NAME = "ADHD-200"
LABEL1, LABEL2 = "Control", "ADHD"

PREPROCESS_TEMPLATE = "AAL116"
TASK_DESC = "interpreting functional connectivity patterns and reasoning about the most likely diagnostic group"
N_SUBJECTS = 768
MAX_NEW_TOKENS = 800

In [5]:
# with open(STRUCTURE_PROMPTS_JSON) as f:
with open(ZSCORED_NUMBERS) as f:
# with open(EDGE_LIST_NUMBERS) as f:
    structure_prompts_all = json.load(f)

if os.path.exists(ROI_NAMES_JSON):
    with open(ROI_NAMES_JSON) as f:
        roi_names = json.load(f)
    print(f"Loaded {len(roi_names)} AAL116 ROI names")
else:
    roi_names = None
    print("[warn] aal_roi_names.json not found -- description will omit the ROI-name legend")

if os.path.exists(META_CSV):
    meta = pd.read_csv(META_CSV, dtype={"subject_id": str})
else:
    meta = None

subject_ids = list(structure_prompts_all.keys())[:N_SUBJECTS]
print(f"Loaded {len(structure_prompts_all)} subjects, running on first {len(subject_ids)}")


subject_ids = list(structure_prompts_all.keys())[:N_SUBJECTS]
print(f"Loaded {len(structure_prompts_all)} subjects, running on first {len(subject_ids)}")
brain_graph_texts = dict(structure_prompts_all)
print("\nExample BrainGraph text (subject 1):")
print(brain_graph_texts[subject_ids[0]])

Loaded 116 AAL116 ROI names
Loaded 768 subjects, running on first 768
Loaded 768 subjects, running on first 768

Example BrainGraph text (subject 1):
Edge feature:
[60]:4.12:[110]
[45]:4.10:[75]
[59]:4.06:[110]
[58]:3.64:[110]
[70]:3.56:[110]
[56]:3.51:[59]
[40]:3.47:[67]
[58]:3.44:[91]
[60]:3.42:[91]
[59]:3.39:[91]
[70]:3.28:[91]
[65]:3.22:[88]
[53]:3.10:[59]
[34]:3.05:[45]
[65]:3.00:[87]
[52]:2.94:[110]
[40]:2.94:[51]
[64]:2.91:[110]
[19]:2.91:[72]
[3]:2.87:[85]
[70]:2.87:[92]
[3]:2.84:[88]
[67]:2.82:[106]
[4]:2.82:[28]
[40]:2.81:[52]
[59]:2.81:[92]
[69]:2.80:[90]
[72]:2.80:[87]
[7]:2.75:[82]
[63]:2.72:[73]
[3]:2.71:[87]
[23]:2.69:[37]
[21]:2.69:[92]
[60]:2.66:[70]
[56]:2.66:[60]
[31]:2.65:[45]
[12]:2.59:[86]
[3]:2.58:[35]
[53]:2.58:[110]
[68]:2.57:[104]
[67]:2.57:[104]
[53]:2.56:[60]
[35]:2.55:[87]
[21]:2.55:[54]
[68]:2.55:[91]
[22]:2.54:[112]
[59]:2.53:[70]
[39]:2.51:[54]
[69]:2.51:[97]
[70]:2.50:[113]
[10]:2.48:[78]
[19]:2.47:[71]
[12]:2.47:[84]
[33]:2.45:[86]
[3]:2.44:[41]
[55]:2

## 3. Prompt Template -- Description / Task / Request

In [6]:
def build_stage1_prompt(subject_id, brain_graph_text):
    return f"""Template: This data comes from the {DATASET_NAME} dataset, preprocessed using the {PREPROCESS_TEMPLATE} brain atlas template (116 regions of interest).
Description: The following is a functional connectivity brain graph for one subject, extracted from resting-state fMRI. Nodes represent brain regions, identified only by their numeric index (no region names provided); edges represent the connectivity strength between two regions as a cohort z-score (not a raw correlation). No node-level features are included, only the ranked list of edges.
Task: {TASK_DESC}
Request: Analyze the BrainGraph edge list below. Find the main connectivity patterns and the most important features. Then predict whether the subject belongs to Control or ADHD. Give a confidence score (0–1) for both classes. Your output must strictly follow this JSON structure and nothing else:
{{
"analysis": "a brief description of the overall connectivity pattern observed in the data (1-2 sentences)",
"key_features": ["feature 1", "feature 2", "..."],
"prediction": "{LABEL1} or {LABEL2}",
"class_confidence": {{"{LABEL1}": 0.0,"{LABEL2}": 0.0}}
}}
BrainGraph (Text format):
{brain_graph_text}"""

In [7]:

first_sid = subject_ids[0]
first_prompt = build_stage1_prompt(first_sid, brain_graph_texts[first_sid])
print(f"=== INPUT SANITY CHECK -- Subject {first_sid} ===")
print(first_prompt)
print(f"\n--- prompt length: {len(first_prompt)} chars, "
      f"{len(tokenizer(first_prompt)['input_ids'])} tokens ---")
input_tokens = tokenizer.encode(first_prompt)

total_input_tokens = len(input_tokens)

print(f"Total input tokens: {total_input_tokens}")


=== INPUT SANITY CHECK -- Subject 1018959 ===
Template: This data comes from the ADHD-200 dataset, preprocessed using the AAL116 brain atlas template (116 regions of interest).
Description: The following is a functional connectivity brain graph for one subject, extracted from resting-state fMRI. Nodes represent brain regions, identified only by their numeric index (no region names provided); edges represent the connectivity strength between two regions as a cohort z-score (not a raw correlation). No node-level features are included, only the ranked list of edges.
Task: interpreting functional connectivity patterns and reasoning about the most likely diagnostic group
Request: Analyze the BrainGraph edge list below. Find the main connectivity patterns and the most important features. Then predict whether the subject belongs to Control or ADHD. Give a confidence score (0–1) for both classes. Your output must strictly follow this JSON structure and nothing else:
{
"analysis": "a brief desc

## 4. Generate -- one subject per call, human-readable parsed output

In [8]:
def parse_stage1_json(raw_text):
    # if there's a <think> block, pull it out and use it as the analysis
    think_match = re.search(r"<think>(.*?)</think>", raw_text, re.DOTALL)
    reasoning = think_match.group(1).strip() if think_match else None

    match = re.search(r"\{.*\}", raw_text, re.DOTALL)
    if not match:
        return None
    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

    if not isinstance(parsed, dict):
        return None

    # if the model produced its own "analysis", leave it; otherwise fill from reasoning
    if reasoning and not parsed.get("analysis"):
        parsed["analysis"] = reasoning

    return parsed

In [ ]:
import os

OUTPUT_DIR = "/home/siu856622573/Code/RQ_2_3/top_pos10/qwen2.5-14B-Instruct/"

CHECKPOINT_PATH = os.path.join(
    OUTPUT_DIR, "qwen_pos10_zscore_scn4_checkpoint.json"
)
FINAL_PATH = os.path.join(
    OUTPUT_DIR, "qwen_pos10_zscore_scn4.json"
)
LOG_PATH = os.path.join(
    OUTPUT_DIR, "qwen_pos10_zscore_scn4_errors.log"
)

SAVE_EVERY = 10          # write checkpoint after every N successful subjects
RETRY_ATTEMPTS = 2       # attempts per subject within a single run

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# import gc
# import json
# import os
# import time

# import torch
# from tqdm.auto import tqdm



# def load_checkpoint():
#     if os.path.exists(CHECKPOINT_PATH):
#         with open(CHECKPOINT_PATH, "r") as f:
#             loaded = json.load(f)
#         # Only trust entries that actually completed successfully -- guards
#         # against an old-format checkpoint that might contain error records.
#         loaded = {sid: rec for sid, rec in loaded.items() if rec.get("prediction") is not None}
#         print(f"Resuming: {len(loaded)} subjects already completed successfully.")
#         return loaded
#     return {}


# def save_checkpoint(results, path=CHECKPOINT_PATH):
#     tmp_path = path + ".tmp"
#     with open(tmp_path, "w") as f:
#         json.dump(results, f, indent=2)
#     os.replace(tmp_path, path)  # atomic write


# def log_line(msg):
#     ts = time.strftime("%Y-%m-%d %H:%M:%S")
#     with open(LOG_PATH, "a") as f:
#         f.write(f"[{ts}] {msg}\n")

# # ---------------  qwen3 and deepseek-r1-distill --------------------------
# # def generate_stage1_response_safe(subject_id, brain_graph_text, max_new_tokens=MAX_NEW_TOKENS):
# #     prompt = build_stage1_prompt(subject_id, brain_graph_text)
# #     messages = [{"role": "user", "content": prompt}]
# #     prompt_text = tokenizer.apply_chat_template(
# #         messages, tokenize=False, add_generation_prompt=True,
# #     )
# #     inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

# #     THINK_BUDGET = 500          # tokens allowed for reasoning
# #     ANSWER_BUDGET = max_new_tokens - THINK_BUDGET   # remaining ~300 for the JSON

# #     try:
# #         with torch.no_grad():
# #             # Stage 1: let it think, but cap it
# #             think_ids = model.generate(
# #                 **inputs, max_new_tokens=THINK_BUDGET,
# #                 do_sample=False, pad_token_id=tokenizer.eos_token_id,
# #             )
# #             think_text = tokenizer.decode(
# #                 think_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
# #             )
# #             # force-close thinking if it didn't finish on its own
# #             if "</think>" not in think_text:
# #                 think_text += "\n</think>\n"

# #             # Stage 2: continue from forced-closed think, now answer in JSON
# #             stage2_prompt = prompt_text + think_text
# #             inputs2 = tokenizer(stage2_prompt, return_tensors="pt").to(model.device)
# #             answer_ids = model.generate(
# #                 **inputs2, max_new_tokens=ANSWER_BUDGET,
# #                 do_sample=False, pad_token_id=tokenizer.eos_token_id,
# #             )
# #             answer_text = tokenizer.decode(
# #                 answer_ids[0][inputs2["input_ids"].shape[1]:], skip_special_tokens=True
# #             )
# #         return think_text + answer_text
# #     finally:
# #         del inputs
# #         gc.collect()
# #         torch.cuda.empty_cache()

## -------------------- GPT and DeepSeek --------------------------
# def generate_stage1_response_safe(subject_id, brain_graph_text, max_new_tokens=MAX_NEW_TOKENS, retries=2):
#     prompt = build_stage1_prompt(subject_id, brain_graph_text)  # unchanged prompt
#     for attempt in range(retries + 1):
#         try:
#             response = client.chat.completions.create(
#                 model=MODEL_ID,
#                 messages=[{"role": "user", "content": prompt}],
#                 max_completion_tokens=max_new_tokens,   # was max_tokens
#             )
#             return response.choices[0].message.content
#         except Exception:
#             if attempt == retries:
#                 raise
#             time.sleep(2 * (attempt + 1))


## 
# def generate_stage1_response_safe(subject_id, brain_graph_text, max_new_tokens=MAX_NEW_TOKENS, retries=2):
#     prompt = build_stage1_prompt(subject_id, brain_graph_text)  # unchanged prompt
#     for attempt in range(retries + 1):
#         try:
#             response = client.chat.completions.create(
#                 model=MODEL_ID,
#                 messages=[{"role": "user", "content": prompt}],
#                 max_completion_tokens=max_new_tokens,   # was max_tokens
#             )
#             return response.choices[0].message.content
#         except Exception:
#             if attempt == retries:
#                 raise
#             time.sleep(2 * (attempt + 1))

# def process_one_subject(sid):
#     """Returns a result dict on success, or raises the last exception after
#     exhausting RETRY_ATTEMPTS."""
#     last_exc = None
#     for attempt in range(1, RETRY_ATTEMPTS + 1):
#         try:
#             raw = generate_stage1_response_safe(sid, brain_graph_texts[sid])
#             parsed = parse_stage1_json(raw)

#             gt = int(meta.loc[meta["subject_id"] == sid, "DX_binary"].values[0])
#             ground_truth = "Control" if gt == 0 else "ADHD"

#             if parsed is None:
#                 # Model ran fine but didn't return valid JSON -- this is NOT
#                 # a CUDA error, retrying won't help, log it and give up on
#                 # this subject for this run (still excluded from JSON).
#                 log_line(f"[parse_fail] {sid} (attempt {attempt}): {raw[:300]!r}")
#                 raise ValueError("Could not parse JSON from model output")

#             return {
#                 "subject_id": sid,
#                 "ground_truth": ground_truth,
#                 "prediction": parsed.get("prediction", ""),
#                 "class_confidence": parsed.get("class_confidence", {}),
#                 "analysis": parsed.get("analysis", ""),
#                 "key_features": parsed.get("key_features", []),
#                 # "conclusion": parsed.get("conclusion", ""),
#             }

#         except Exception as e:
#             last_exc = e
#             log_line(f"[error] {sid} (attempt {attempt}/{RETRY_ATTEMPTS}): {repr(e)}")
#             gc.collect()
#             torch.cuda.empty_cache()
#             torch.cuda.synchronize()

#     raise last_exc


In [ ]:
import gc
import json
import os
import time

import torch
from tqdm.auto import tqdm



def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "r") as f:
            loaded = json.load(f)
        # Only trust entries that actually completed successfully -- guards
        # against an old-format checkpoint that might contain error records.
        loaded = {sid: rec for sid, rec in loaded.items() if rec.get("prediction") is not None}
        print(f"Resuming: {len(loaded)} subjects already completed successfully.")
        return loaded
    return {}


def save_checkpoint(results, path=CHECKPOINT_PATH):
    tmp_path = path + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(results, f, indent=2)
    os.replace(tmp_path, path)  # atomic write


def log_line(msg):
    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    with open(LOG_PATH, "a") as f:
        f.write(f"[{ts}] {msg}\n")

def generate_stage1_response_safe(subject_id, brain_graph_text, max_new_tokens=MAX_NEW_TOKENS):
    prompt = build_stage1_prompt(subject_id, brain_graph_text)
    messages = [{"role": "user", "content": prompt}]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    try:
        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False, pad_token_id=tokenizer.eos_token_id,
            )
        answer_text = tokenizer.decode(
            output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )
        return answer_text
    finally:
        del inputs
        gc.collect()
        torch.cuda.empty_cache()


def process_one_subject(sid):
    """Returns a result dict on success, or raises the last exception after
    exhausting RETRY_ATTEMPTS."""
    last_exc = None
    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            raw = generate_stage1_response_safe(sid, brain_graph_texts[sid])
            parsed = parse_stage1_json(raw)

            gt = int(meta.loc[meta["subject_id"] == sid, "DX_binary"].values[0])
            ground_truth = "Control" if gt == 0 else "ADHD"

            if parsed is None:
                # Model ran fine but didn't return valid JSON -- this is NOT
                # a CUDA error, retrying won't help, log it and give up on
                # this subject for this run (still excluded from JSON).
                log_line(f"[parse_fail] {sid} (attempt {attempt}): {raw[:300]!r}")
                raise ValueError("Could not parse JSON from model output")

            return {
                "subject_id": sid,
                "ground_truth": ground_truth,
                "prediction": parsed.get("prediction", ""),
                "class_confidence": parsed.get("class_confidence", {}),
                "analysis": parsed.get("analysis", ""),
                "key_features": parsed.get("key_features", []),
                # "conclusion": parsed.get("conclusion", ""),
            }

        except Exception as e:
            last_exc = e
            log_line(f"[error] {sid} (attempt {attempt}/{RETRY_ATTEMPTS}): {repr(e)}")
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

    raise last_exc


## Sanity check

In [12]:
# ---------------------------------------------------------------------
# Sanity check — run this before the main inference loop
# ---------------------------------------------------------------------
assert len(subject_ids) == N_SUBJECTS, f"Expected {N_SUBJECTS} subjects, got {len(subject_ids)}"
assert all(sid in brain_graph_texts for sid in subject_ids), "Missing brain_graph_texts for some subject_ids"
assert meta is not None, "meta not loaded"
assert set(subject_ids) <= set(meta["subject_id"]), "Some subject_ids missing from meta"

gt_counts = meta.loc[meta["subject_id"].isin(subject_ids), "DX_binary"].value_counts()
print(f"Ground truth check: Control={gt_counts.get(0,0)}, ADHD={gt_counts.get(1,0)} "
      f"(expected 488 / 280)")

# don't silently overwrite a different scenario's completed run
print(FINAL_PATH)
# one live generation, checked against the schema the prompt actually asks for
_test_sid = subject_ids[0]
_raw = generate_stage1_response_safe(_test_sid, brain_graph_texts[_test_sid])
_parsed = parse_stage1_json(_raw)
assert _parsed is not None, f"Sanity-check generation failed to parse JSON:\n{_raw[:500]}"
_expected_keys = {"analysis", "key_features", "prediction", "class_confidence"}
_missing = _expected_keys - _parsed.keys()
assert not _missing, f"Parsed JSON missing expected keys: {_missing}\nGot keys: {list(_parsed.keys())}\nRaw output:\n{_raw[:800]}"
assert _parsed["prediction"] in {"Control", "ADHD"}, f"Unexpected prediction value: {_parsed['prediction']}"

print(f"Sanity check passed on subject {_test_sid} — prediction={_parsed['prediction']}, "
      f"keys OK, {len(subject_ids)} subjects ready, output -> {FINAL_PATH}")

      
sid = subject_ids[0]  # or any specific ID, e.g. "1018959"

raw = generate_stage1_response_safe(sid, brain_graph_texts[sid])
print(repr(raw))  # exact raw text, for debugging if parsing fails

parsed = parse_stage1_json(raw)

if parsed is None:
    print("\nPARSE FAILED -- raw text above was not valid JSON.")
else:
    print("\nPARSED SUCCESSFULLY:")
    print(f"  prediction:       {parsed.get('prediction')}")
    print(f"  class_confidence: {parsed.get('class_confidence')}")
    print(f"  key_features:     {parsed.get('key_features')}")
    print(f"  analysis:         {parsed.get('analysis')}")
    # print(f"  conclusion:       {parsed.get('conclusion')}")

Ground truth check: Control=488, ADHD=280 (expected 488 / 280)
/home/siu856622573/Code/RQ_2_3/top_pos10/phi/phi_pos10_zscore_scn4.json


/home/siu856622573/.conda/envs/py10_roy/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Sanity check passed on subject 1018959 — prediction=ADHD, keys OK, 768 subjects ready, output -> /home/siu856622573/Code/RQ_2_3/top_pos10/phi/phi_pos10_zscore_scn4.json
'```json\n{\n  "analysis": "The connectivity pattern shows strong connections between certain nodes, with high z-scores indicating significant connectivity. Notably, nodes 110, 91, and 59 frequently appear with high z-scores, suggesting key hubs in the network.",\n  "key_features": [\n    "High connectivity strength between nodes 110, 91, and 59",\n    "Presence of both high and low z-score connections",\n    "Clusters of nodes with strong connectivity"\n  ],\n  "prediction": "ADHD",\n  "class_confidence": {\n    "Control": 0.3,\n    "ADHD": 0.7\n  }\n}\n```\n\n### Explanation:\n- **Analysis**: The connectivity graph shows several nodes with high z-scores, indicating strong connections. Nodes 110, 91, and 59 are particularly prominent, suggesting they are central in the network.\n- **Key Features**: The presence of stro

In [13]:
sid = subject_ids[122]  # or any specific ID, e.g. "1018959"

raw = generate_stage1_response_safe(sid, brain_graph_texts[sid])
print(repr(raw))  # exact raw text, for debugging if parsing fails

parsed = parse_stage1_json(raw)

if parsed is None:
    print("\nPARSE FAILED -- raw text above was not valid JSON.")
else:
    print("\nPARSED SUCCESSFULLY:")
    print(f"  prediction:       {parsed.get('prediction')}")
    print(f"  class_confidence: {parsed.get('class_confidence')}")
    print(f"  key_features:     {parsed.get('key_features')}")
    print(f"  analysis:         {parsed.get('analysis')}")
    # print(f"  conclusion:       {parsed.get('conclusion')}")

'```json\n{\n  "analysis": "The connectivity pattern shows strong connections between certain nodes, with the highest z-scores indicating significant connectivity. Notably, nodes 11, 4, and 27 have multiple high-strength connections, suggesting a central role in the network.",\n  "key_features": [\n    "High connectivity strength between nodes 11, 4, and 27",\n    "Multiple connections with z-scores above 2.5",\n    "Presence of both high and low z-score connections, indicating a diverse connectivity pattern"\n  ],\n  "prediction": "ADHD",\n  "class_confidence": {\n    "Control": 0.3,\n    "ADHD": 0.7\n  }\n}\n```\n\nThis analysis is based on the observation that the connectivity pattern, characterized by strong connections between specific nodes, aligns more closely with typical ADHD connectivity profiles, which often show distinct patterns of connectivity strength. The confidence score reflects a higher likelihood of the subject belonging to the ADHD group.'

PARSED SUCCESSFULLY:
  p

## Inference

In [ ]:
# ---------------------------------------------------------------------
# Resume from checkpoint, figure out what's left to do
# ---------------------------------------------------------------------
results = load_checkpoint()
remaining_ids = [sid for sid in subject_ids if sid not in results]
print(f"Total subjects: {len(subject_ids)} | already done: {len(results)} | remaining: {len(remaining_ids)}")

n_success, n_parse_fail, n_error = 0, 0, 0
start_time = time.time()

with open(LOG_PATH, "a") as log_f:
    pbar = tqdm(remaining_ids, desc="Subjects", unit="subj")
    for sid in pbar:
        try:
            raw = generate_stage1_response_safe(sid, brain_graph_texts[sid])
            parsed = parse_stage1_json(raw)

            gt = int(meta.loc[meta["subject_id"] == sid, "DX_binary"].values[0])
            ground_truth = "Control" if gt == 0 else "ADHD"

            if parsed:
                results[sid] = {
                    "subject_id": sid,
                    "ground_truth": ground_truth,
                    "prediction": parsed.get("prediction", ""),
                    "class_confidence": parsed.get("class_confidence", {}),
                    "analysis": parsed.get("analysis", ""),
                    "key_features": parsed.get("key_features", []),
                    # "conclusion": parsed.get("conclusion", ""),
                }
                n_success += 1
            else:
                # Model ran fine, just didn't return valid JSON -- this
                # counts as "done", NOT retried. Raw output is kept so
                # you can inspect/re-parse it later if needed.
                results[sid] = {
                    "subject_id": sid,
                    "ground_truth": ground_truth,
                    "raw_output": raw,
                    "parsed": None,
                }
                n_parse_fail += 1
                log_f.write(f"[parse_fail] {sid}: {raw[:300]!r}\n")
                log_f.flush()

        except Exception as e:
            # A real crash (CUDA error, etc.) -- NOT saved to results, stays
            # "not done", so this exact subject is retried the next time
            # this cell runs.
            n_error += 1
            log_f.write(f"[error] {sid}: {repr(e)}\n")
            log_f.flush()
            gc.collect()
            torch.cuda.empty_cache()

        done = len(results)
        pbar.set_postfix(done=done, ok=n_success, parse_fail=n_parse_fail, err=n_error)

        if done % SAVE_EVERY == 0:
            save_checkpoint(results)

# Final save + promote checkpoint to the "done" filename once everything ran
save_checkpoint(results)
elapsed = time.time() - start_time
print(f"\nFinished this run in {elapsed/60:.1f} min.")
print(f"Total saved: {len(results)}/{len(subject_ids)}  "
      f"(ok={n_success}, parse_fail={n_parse_fail}, error={n_error})")

if len(results) == len(subject_ids):
    os.replace(CHECKPOINT_PATH, FINAL_PATH)
    print(f"All subjects done -> {FINAL_PATH}")
else:
    print(f"Not all subjects done yet -- re-run this cell to resume from {CHECKPOINT_PATH}")


In [ ]:
import json, os

VALID_LABELS = {"ADHD", "Control"}
print(FINAL_PATH)

def print_stats(data):
    n = len(data)

    n_no_pred = sum(1 for r in data.values() if r.get("prediction") is None)
    malformed_entries = [
        (sid, r.get("prediction"))
        for sid, r in data.items()
        if r.get("prediction") is not None and r.get("prediction") not in VALID_LABELS
    ]
    n_malformed_pred = len(malformed_entries)

    # scorable = BOTH ground_truth and prediction are exactly "ADHD" or "Control"
    scorable = [
        r for r in data.values()
        if r.get("ground_truth") in VALID_LABELS and r.get("prediction") in VALID_LABELS
    ]
    n_scorable = len(scorable)

    n_gt_adhd_all = sum(1 for r in data.values() if r.get("ground_truth") == "ADHD")
    n_gt_control_all = sum(1 for r in data.values() if r.get("ground_truth") == "Control")

    n_pred_adhd = sum(1 for r in scorable if r["prediction"] == "ADHD")
    n_pred_control = sum(1 for r in scorable if r["prediction"] == "Control")

    n_correct = sum(1 for r in scorable if r["prediction"] == r["ground_truth"])

    # Per-class, denominator restricted to scorable subjects only
    scorable_gt_adhd = [r for r in scorable if r["ground_truth"] == "ADHD"]
    scorable_gt_control = [r for r in scorable if r["ground_truth"] == "Control"]
    n_adhd_correct = sum(1 for r in scorable_gt_adhd if r["prediction"] == "ADHD")
    n_control_correct = sum(1 for r in scorable_gt_control if r["prediction"] == "Control")

    print(f"Total subjects in file:              {n}")
    print(f"  -- no prediction (parse failure):  {n_no_pred}")
    print(f"  -- malformed prediction (neither ADHD nor Control exactly): {n_malformed_pred}")
    if malformed_entries:
        print(f"     malformed subject_ids and their prediction values:")
        for sid, val in malformed_entries:
            print(f"       {sid}: {val!r}")
    print(f"  -- scorable (valid ground_truth AND valid prediction): {n_scorable}")
    print()
    print(f"Ground truth (all {n} subjects) -- ADHD: {n_gt_adhd_all} | Control: {n_gt_control_all}")
    print(f"Prediction (scorable subjects only, n={n_scorable}) -- ADHD: {n_pred_adhd} | Control: {n_pred_control}")
    print()
    if n_scorable:
        print(f"Accuracy (scorable only): {n_correct}/{n_scorable} = {n_correct / n_scorable:.2%}")
    else:
        print("Accuracy: n/a (no scorable subjects)")

    print()
    print("Correctly predicted, by class (denominator = scorable subjects of that class only):")
    if scorable_gt_adhd:
        print(f"  ADHD    correctly predicted: {n_adhd_correct}/{len(scorable_gt_adhd)} = {n_adhd_correct / len(scorable_gt_adhd):.2%}")
    else:
        print("  ADHD    correctly predicted: n/a")
    if scorable_gt_control:
        print(f"  Control correctly predicted: {n_control_correct}/{len(scorable_gt_control)} = {n_control_correct / len(scorable_gt_control):.2%}")
    else:
        print("  Control correctly predicted: n/a")


if os.path.exists(FINAL_PATH):
    with open(FINAL_PATH) as f:
        data = json.load(f)
    print("RUN COMPLETE.")
    print_stats(data)

else:
    print("No output file found yet -- nothing processed.")
